## 5.0 ConversationBufferMemory
- Save whole conversations.
- 대화가 길어질수록 메모리도 커져 비효율적
- 모델 자체에는 메모리가 없기 때문에, 모델에 요청을 보낼 때마다 대화 전체를 같이 보내야 한다.
- 텍스트 자동완성 시 사용한다.

In [ ]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory() # default parameter: return_messages=False. for return text.
memory.save_context({"input":"Hi!", "output":"How are you?"})
memory.load_memory_variables({})

# returns {'history': 'Human: Hi!\nAI: How are you?'}

In [ ]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(return_messages=True) # for chat model
memory.save_context({"input":"Hi!", "output":"How are you?"})
memory.load_memory_variables({})

# returns {'history': [HumanMessage(content='Hi!'), AIMessage(content='How are you?')]}

## 5.1 ConversationBufferWindowMemory
- 최근 n개의 메시지까지만 저장
- 장점: 메모리를 특정 크기로 유지할 수 있다.
- 단점: 최근 대화에만 집중한다.

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(
    return_messages=True,
    k=2, # Buffer window size: 저장할 메시지 갯수
)

def add_message(input, output):
    memory.save_context({"input":input, "output":output})

add_message(1,1)
add_message(2,2)
add_message(3,3)

memory.load_memory_variables({})

## 5.2 ConversationSummaryMemory
- 대화를 요약해서 저장한다.
- llm을 사용하기 때문에 돈이 든다.
- 초반에는 토큰을 더 차지하지만, 대화가 길어질수록 덜 차지한다.

In [1]:
def add_message(input, output):
    memory.save_context({"input":input, "output":output})

def get_history():
    memory.load_memory_variables({})

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryMemory

llm = ChatOpenAI(temperature=0.1)
memory = ConversationSummaryMemory(llm=llm)

add_message("Hi I'm Nicolas, I live in South Korea", "Wow that is so cool!")
add_message("South Kddorea is so pretty", "I wish I could go!!!")

get_history()

# result: {'history': 'summary'}

## 5.3 ConversationSummaryBufferMemory
- ConversationSummaryMemory + ConversationBufferMemory
- 메시지 수를 세서, 설정한 한도를 넘으면 오래된 메시지를 요약한다.
- API를 사용하기 때문에 돈이 든다.

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryBufferMemory

llm = ChatOpenAI(temperature=0.1)
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=150, # 메시지가 요약되기 전 최대 토큰 값.
    return_messages=True,
)

add_message("Hi I'm Nicolas, I live in South Korea", "Wow that is so cool!")
add_message("South Korea is so pretty", "I wish I could go!!!")
add_message("How far is Korea from Argentina?", "I don't know! Super far!")
add_message("How far is Brazil from Argentina?", "I don't know! Super far!")
get_history()

"""
result: 
{'history': [SystemMessage(content='The human introduces themselves as Nicolas and mentions that they live in South Korea. The AI responds by expressing excitement and finding it cool.'),
  HumanMessage(content='South Korea is so pretty'),
  AIMessage(content='I wish I could go!!!'),
...}  
"""

## 5.4 ConversationKGMemory(Conversation Knowledge Graph Memory)
- 개요만 뽑아내는 요약본

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationKGMemory

llm = ChatOpenAI(temperature=0.1)

memory = ConversationKGMemory(
    llm=llm,
    return_messages=True,
)

def add_message(input, output):
    memory.save_context({"input":input, "output":output})

add_message("Nicolas likes kimchi", "Wow that is so cool!")
memory.load_memory_variables({"input":"what does nicolas like"})

'''
{'history': [SystemMessage(content='On Nicolas: Nicolas is a person. Nicolas lives in South Korea. Nicolas likes kimchi.')]}
'''

### ConversationTokenBufferMemory
- ConversationBufferWindowMemory와 유사
- k(interaction) 대신 token 수로 한계 설정

### Entity
- 대화 중에 entitiy를 추출한다.

### [Memory Integrations](https://python.langchain.com/docs/integrations/memory/)

## 5.5 Memory on LLMChain
1. LLMChain: off-the-shelf(general purpose) chain

In [ ]:
# 1. Text based memory on LLMChain
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    memory_key="chat_history" # template의 같은 변수 안에 집어 넣기
)

template = """
You are a helpful AI talking to human.

{chat_history}
Human: {question}
You:
"""

chain = LLMChain(
    llm=llm,
    memory=memory,
    prompt=PromptTemplate.from_template(template),
    verbose=True, # prompt log 확인
)

chain.predict(question="My name is Nico")
chain.predict(question="I live in Seoul")
chain.predict(question="What is my name?")

## 5.6 Chat Based Memory on LLMChain

In [ ]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains import LLMChain

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    return_messages=True,
    memory_key="chat_history",
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI talking to human."),
    MessagesPlaceholder(variable_name="chat_history"), # 불특정 다수의 예측할 수 없는 양의 메시지를 넣을 때 사용한다.
    ("human","{question}"),
])

chain = LLMChain(
    llm=llm,
    memory=memory,
    verbose=True,
    prompt=prompt,
)

## 5.7 LCEL Based Memory
2. Memory on Custom Chain using LCEL

In [ ]:
# 1. 첫 번째 방법. chat_history를 chain.invoke 할 때 항상 넣어줘야 함. 두 번째 방법에서 개선
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryBufferMemory
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    return_messages=True,
    memory_key="chat_history",
)

prompt = ChatPromptTemplate([
    ("system","You are a helpful AI talking to human."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

chain = prompt | llm
chain.invoke({
    "chat_history": memory.load_memory_variables({})["chat_history"],
    "question": "My name is Nico"
})

In [ ]:
# 2. 두 번쨰 방법. RunnablePassthrough.assign을 사용해서 prompt를 format하기 전에 함수를 실행하여 chat_history 변수에 할당한다.
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryBufferMemory
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema.runnable import RunnablePassthrough

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    return_messages=True,
    # default_memory_key="history",
)

prompt = ChatPromptTemplate([
    ("system","You are a helpful AI talking to human."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

# def load_memory(input): 또는 아래와 같이 input 무시
def load_memory(_):
    return memory.load_memory_variables({})["history"]

                                   #3      #2           | #4
chain = RunnablePassthrough.assign(history=load_memory) | prompt | llm

def invoke_chain(question):
    #1
    result = chain.invoke(
        {"question":question}
    )
    memory.save_context(
        {"input":question},
        {"output": result.content},
    )